In [ ]:
# Remove all the warnings

import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from litellm import completion

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
prompt = "Explain what is Agentic AI in one line"

# list of providers
providers = [
    ("Open AI", "gpt-40-mini"),
    ("Groq", "groq/llama-3.3-70b-versatile"),
    ("Anthropic", "claude-3-5-haiku-20241022"),
    ("Gemini", "gemini/gemini-1.5-flash"),
]

# one function call and multiple providers - but these are not automatic fallback
for label, model in providers:
    try:
        r = completion(model=model, messages=[ {"role":"user", "content":prompt }])
        print(f"Using {model} : Ans : {r.choices[0].message.content[:100]}")
    except Exception as e:
        print(f"{type(e)}.__name__")



In [ ]:
# Automatic fallback if any of the model is down
from litellm import completion

response = completion(
    model = "gemini/gemini-1.5-flash",
    messages=[{"role":"user", "content": prompt}],
    fallbacks=[
        "gpt-4o-mini",
        "groq/llama-3.7-70b-versatile",
    ]
)

print(f"Answer is : {response.choices[0].message.content[:200]}")
print(f"The above answer is from : {response.model} model")

In [ ]:
# It's importance to see where calls are being made
from litellm import completion, completion_cost

response = completion(
    model = "gemini/gemini-1.5-flash",
    messages=[{"role":"user", "content": prompt}],
    fallbacks=[
        "gpt-4o-mini",
        "groq/llama-3.7-70b-versatile",
    ]
)

cost = completion_cost(completion_response=response)

print(f"Response : {response.choices[0].message.content}")
print(f"Input tokens : {response.usage.prompt_tokens}")
print(f"Output tokens : {response.usage.completion_tokens}")
print(f"Total cost : {cost}")

In [ ]:
# If you have the same question why should you pay again? use caching
import time
from litellm.caching import Cache

litellm.cache = Cache(type="local")

response = completion(
    model = "gemini/gemini-1.5-flash",
    messages=[{"role":"user", "content": prompt}],
    caching=True, 
    fallbacks=[
        "gpt-4o-mini",
        "groq/llama-3.7-70b-versatile",
    ]
)

print(f"Response : {response.choices[0].message.content}")
print(f"Input tokens : {response.usage.prompt_tokens}")
print(f"Output tokens : {response.usage.completion_tokens}")

In [ ]:
# Right model for right JOB
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params":{
            "model": "groq/llama-3.7-70b-versatile"
            "api_key": os.getenv["GROQ_API_KEY"]
        }
    },
    {
        "model_name": "smart-coding",
        "litellm_params":{
            "model": "claude-3-5-haiku-20241022"
            "api_key": os.getenv["ANTHROPIC_API_KEY"]
        }
    },
]

router = Router(model_list=model_list)

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role":"user", "content": prompt}]
)
code_response = router.completion(
    model="smart-coding",
    messages=[{"role":"user", "content": prompt}]
)

In [ ]:
# Balance the requests across the providers
import os
from litellm import Router

model_list = [
    {
        "model_name": "gpts",
        "litellm_params":{
            "model": "groq/llama-3.7-70b-versatile"
            "api_key": os.getenv["GROQ_API_KEY"]
        },
        "model_info":{"id":"groq-llama-70b"}
    },
    {
        "model_name": "gpts-coding",
        "litellm_params":{
            "model": "gpt-4o"
            "api_key": os.getenv["ANTHROPIC_API_KEY"]
        },
        "model_info":{"id":"openai-gpt4o"}
    },
]


router = Router(
    model_list=model_list,
    routing_strategy="simple-shuffle" # shuffle
)

